# S0/S1 sanity ladder: oracle query path and non-amortised shift ceiling (A→B architecture)

## Description

<!-- Written before the first run (2026-10-28); kept up to date. Design: docs/design_2026-10-28_A_to_B_architecture.md §8. Issue #16. -->

**Question.** The proposed architecture moves A's points and queries *into* B's domain and reads B through the frozen
single-task PFN (the query path $Q_B$, B's per-layer K/V cached once). Two questions are answered before anything is trained:

- **S0, plumbing.** Is the *oracle particle* $Q_B(T(x^*))$ (B queried at the true transform, A unused) as good as the
  privileged teacher, and much better than the A-only PFN? This validates the query path, caching and scale handling
  ($h = \mathrm{id}$: $Q_B$'s y-scale is A's).
- **S1, non-amortised ceiling.** With $T(x) = x + s$ unknown, how much of the oracle's value can a Bayes learner recover
  from A's few points? Posterior over $s$ on a grid, its MAP error, entropy, **fraction of unimodal posteriors** (when is one
  hypothesis enough), and whether gradient descent on $s$ through $Q_B$ finds the MAP (is the landscape navigable).

**Setup.** Frozen `pfn_large` (pfn-repro v0.1.0, the variable-d checkpoint, trained on contexts $n \le 100$), no training.
$f \sim$ BNN prior on $[0,1]^d$ (the PFN's training prior and ECDF scale), $h = \mathrm{id}$, $T(x) = x + s$,
$s \sim U(-0.2, 0.2)^d$, $d \in \{1, 2\}$. B: uniform design, $N_B = 80$ (main arm; $N_B + n_A \le 90$). A: train and query
inputs drawn so that $T(x) \in [0,1]^d$; $n_A \in \{0, 1, 2, 5, 10\}$ (nested), 50 held-out queries. 100 pairs per stratum:
`shift`; `identity` ($s = 0$, sanity); `unrelated` (B observes an independent function: negative transfer). Split 0 (pilot).
One member per $(d, N_B)$; $N_B = 200$ (the design note's value) is run as an **out-of-range probe** only.

| predictor | context | query | role |
|---|---|---|---|
| `a_only` | $A_{<n}$ | $x^*$ | lower bound |
| `oracle_q` | $B$ | $T(x^*)$ | oracle particle (S0 (b)) |
| `teacher_a` | $A_{<n} \cup B_{\mathrm{in}A}$ (B points with $x_b - s \in [0,1]^d$) | $x^*$ | A-space teacher (S0 (c)) |
| `teacher_b` | $B \cup A_{\mathrm{in}B,<n}$ | $T(x^*)$ | B-space teacher (§7.2) |
| `naive` | $B \cup A_{<n}$ | $x^*$ | pooling without registration |
| `grid_pseudo` | $B$ | $x^* + s$, mixed over the grid posterior | S1, the architecture's pseudo-likelihood $\sum_a \log Q_B(y_a \mid x_a + s)$ |
| `map_pseudo` | $B$ | $x^* + \hat s_{\mathrm{MAP}}$ | S1, one particle at the MAP (what S2 aims at) |
| `grid_exact` | $B \cup A_{\mathrm{in}B,<n}(s)$ | $x^* + s$, chain-rule posterior | S1, the exact Bayes learner for the shift model |
| `bayes_null` | mixture of `grid_exact` and `a_only`, prior 1/2 | | S1 with the null hypothesis ("unrelated") |

Grid: 201 points ($d=1$, step 0.002), $41^2$ ($d=2$, step 0.01), uniform prior (= the generating prior); $x + s$ clipped to
$[0,1]^d$. GD: Adam (lr 0.01, 200 steps) on the pseudo log-likelihood from $s = 0$, $s$ projected to the prior box; success =
its log-likelihood within 0.1 nats of the grid MAP's.

*Changes after a first look at the $d = 1$ members of the first launch (2026-10-28 14:35; that sweep was stopped and
relaunched, nothing from it is reported):* the posterior entropy was NaN for sharp posteriors (float32 underflow; now
float64), and GD is also run from 8 starts (0 + 7 uniform in the prior box; best by log-likelihood), descriptive only,
because the single start reached the MAP in only about half the pairs. The GO / NO-GO rules below are unchanged.

**What to look for.** (1) NLL vs $n_A$ per predictor, stratum and $d$ (mean ± SE over pairs: the MC error);
(2) $d=1$ examples: $f_A$, the oracle particle and the A-only PFN; (3) posteriors over $s$ at $n_A = 1, 2, 5, 10$;
(4) MAP error, entropy, unimodal fraction vs $n_A$; (5) GD success and trajectories on the likelihood landscape;
(6) the $N_B = 200$ probe against $N_B = 80$.

**GO / NO-GO** (*drafted by Claude, 2026-10-28, before the first run*; shift stratum, $N_B = 80$, paired over pairs):

- **S0 PASS** if for both $d$, at every $n_A \in \{0, 1, 2\}$:
  (P1) NLL(`a_only`) − NLL(`oracle_q`) ≥ 0.5 nats/query with the 95% CI's lower bound > 0 ("≪");
  (P2) |NLL(`oracle_q`) − NLL(`teacher_b`)| ≤ 0.1 and |NLL(`oracle_q`) − NLL(`teacher_a`)| ≤ 0.25 ("≈"; `teacher_a` sees
  fewer B points). S0 FAIL otherwise. Exact identities ($n_A = 0$: `teacher_b` = `oracle_q`; the first chain-rule term =
  the pseudo-likelihood) are unit tests (`tests/unit/test_ladder.py`).
- **S1 → build S2?** At $n_A = 5$: **GO** if for both $d$ `grid_pseudo` closes ≥ 50% of the gap `a_only` → `oracle_q`
  (ratio of means) *and* the unimodal fraction (pseudo posterior) is ≥ 0.6; **NO-GO** if for either $d$ it closes < 20%
  (the location signal is not in A's few points); **INCONCLUSIVE** otherwise.
- Descriptive, not gated: GD success rate (< 50% means S2 needs refinement steps or several starts), the
  pseudo-vs-exact gap (cost of the independence assumption, §6), the unrelated stratum's harm and `bayes_null`'s recovery.

**Limits.** `pfn_large` is valid for $n \le 100$, $d \le 6$: the $N_B = 200$ probe is outside and is not used for any verdict.
Shift-only $T$, $h = \mathrm{id}$, uniform B, no residual: the easiest rung by design. Wrong hypotheses push $x + s$ outside
the box, where the input is clipped (the frozen PFN never saw $x \notin [0,1]$). The grid prior equals the generating prior,
which favours the Bayes learner. Pilot split only; nothing here is tuned on split 1.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from meta_pfn.analysis import load_snapshot, show_header
from meta_pfn.analysis import ladder as LA

SNAPSHOT = "results/ladder/pilot_20261028-143711"
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot
T = {k: LA.with_cell(snap.table(k), snap.path) for k in ("nll", "post", "gd", "ex_post", "ex_gd", "ex_points")}
curve = LA.with_cell(snap.table("ex_curve"), snap.path)
nll, post, gd = T["nll"], T["post"], T["gd"]
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "lines.linewidth": 2})
print(snap.manifest["status_counts"], "| pairs per (member, kind):",
      nll.groupby(["member", "kind"]).pair.nunique().unique())

## Results

Aggregates use DONE members only; the status counts above say what is failed / started / missing. Error bars and bands are ±1 SE over pairs (the Monte-Carlo error of every mean, the oracle's included).

### S0: the query path (NLL per query vs $n_A$, $N_B = 80$)

Rows: $d$; columns: the shift stratum, the identity sanity stratum and the unrelated (negative-transfer) stratum.

In [ ]:
def nll_panels(preds, n_b=80, title=""):
    s = LA.nll_summary(nll)
    kinds = ["shift", "identity", "unrelated"]
    ds = sorted(s.d.unique())
    fig, axs = plt.subplots(len(ds), 3, figsize=(13, 3.6 * len(ds)), squeeze=False)
    for r, d in enumerate(ds):
        for c, kind in enumerate(kinds):
            ax = axs[r, c]
            for p in preds:
                g = s[(s.d == d) & (s.n_b == n_b) & (s.kind == kind) & (s.predictor == p)].sort_values("n_a")
                ax.plot(g.n_a, g["mean"], "-o", ms=4, color=LA.COLORS[p], label=p)
                ax.fill_between(g.n_a, g["mean"] - g.se, g["mean"] + g.se, color=LA.COLORS[p], alpha=0.15, lw=0)
            ax.set_title(f"d = {d}, {kind}")
            ax.set_xlabel("$n_A$")
            if c == 0:
                ax.set_ylabel("NLL per query (nats)")
    axs[0, -1].legend(loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False)
    fig.suptitle(title)
    fig.tight_layout()
    return fig


nll_panels(LA.S0_PRED, title="S0: A-only vs the oracle particle Q_B(T(x*)) vs the teachers");

**How to read.** Each line is the mean, over the 100 pairs of the stratum, of the held-out NLL per A query:
$-\log p(y^* \mid \cdot)$ under the predictor's bar distribution, in nats; **lower is better**. $y$ lives on $[0, 1]$ (the
ECDF scale), so densities above 1 are normal and NLL can be negative: $-4$ means a density of $e^4 \approx 55$ at the
observation, i.e. a predictive about 0.02 wide. A difference of 1 nat = a factor $e$ in density at the observed value.
The x-axis is $n_A$, A's labelled points in context (nested prefixes of one design, so the curves are paired across
$n_A$). Bands: ±1 SE over pairs. Each panel has its own y-scale.

- `oracle_q` is flat: it reads only B (at $x^* + s$) and ignores A. `teacher_b` lies on top of it, hiding the blue line.
- **Shift column:** the vertical gap between grey (A only) and blue/green is what B is worth when the registration is
  known: the ceiling for everything later. Red (naive pooling) *above* grey: adding B unregistered hurts.
- **Identity column (sanity):** with $s = 0$ every B-based predictor coincides, naive included.
- **Unrelated column (negative transfer):** B-based predictors are catastrophically confident and wrong (~+12 nats);
  the teachers recover slowly because A's points in the context gradually override B's.

In [ ]:
verdict0, t0 = LA.s0_verdict(nll)
print("S0 (drafted rule):", verdict0, "| largest SE of any mean NLL:", round(LA.mc_error(nll), 3))
t0.round(3)

### S0 examples ($d = 1$, shift)

The true $f_A$, the oracle particle (B queried at $x + s$; mean and 10–90% band) and the A-only PFN with $n_A = 5$ (black dots: A's first 5 points).

In [ ]:
ex = curve[(curve.kind == "shift") & (curve.n_b == 80)]
pairs = sorted(ex.pair.unique())
fig, axs = plt.subplots(1, len(pairs), figsize=(4.4 * len(pairs), 3.4), squeeze=False)
for ax, p in zip(axs[0], pairs):
    e = ex[ex.pair == p]
    f = e[e.model == "f_A"]
    ax.plot(f.x, f["mean"], color="black", lw=1.2, label="true $f_A$")
    for m in ("oracle_q", "a_only"):
        g = e[e.model == m]
        ax.plot(g.x, g["mean"], color=LA.COLORS[m], label=m + (" ($n_A$=5)" if m == "a_only" else ""))
        ax.fill_between(g.x, g.q10, g.q90, color=LA.COLORS[m], alpha=0.15, lw=0)
    pts = T["ex_points"]
    a = pts[(pts.kind == "shift") & (pts.n_b == 80) & (pts.pair == p) & (pts.set == "A")].head(5)
    ax.plot(a.x0, a.y, "o", color="black", ms=6)
    s_true = post[(post.kind == "shift") & (post.n_b == 80) & (post.d == 1) & (post.pair == p)].s_true.iloc[0][0]
    ax.set_title(f"pair {p}, s = {s_true:+.3f}")
    ax.set_xlabel("x (A's coordinates)")
axs[0, 0].set_ylabel("y (ECDF scale)")
axs[0, -1].legend(frameon=False, fontsize=8)
fig.tight_layout()

**How to read.** In A's coordinates. Blue = B read through the PFN at $x + s$ (the oracle particle); it should
trace the black truth everywhere, including far from A's points. Grey = A-only PFN with the 5 black points: good near
them, wide (10–90% band) elsewhere. Wiggles that B gets and A-only cannot (pair 2) are exactly the transferable part.

### S1: the non-amortised ceiling (NLL per query vs $n_A$, $N_B = 80$)

`grid_pseudo` is what the architecture's likelihood can reach with a full grid of particles; `map_pseudo` is one particle at the MAP; `grid_exact` is the exact Bayes learner for the shift model; `bayes_null` adds the hypothesis "unrelated".

In [ ]:
nll_panels(LA.S1_PRED, title="S1: grid-Bayes over the shift s");

**How to read.** Same axes as S0; now $s$ is *unknown* and inferred from A's points on a grid (uniform prior).

- **$n_A = 0$:** `grid_*` average B's prediction over all shifts (the prior predictive). It is already better than A
  only, because B tells the range and shape of values. `map_pseudo` is very bad there: with a flat posterior the argmax
  is an arbitrary grid point ($s = -0.2$), i.e. one particle committed to a random hypothesis. **Where yellow joins
  orange tells you when a single hypothesis is enough** (about $n_A = 5$).
- **Orange/purple → blue:** how fast the shift is identified. The orange–purple gap is the cost of the pseudo-likelihood
  (A's points independent given B) against the exact chain rule.
- **Unrelated:** `grid_pseudo` gets *worse* with $n_A$ (it grows confident in some wrong shift); `grid_exact` improves
  because A's points enter its context. Pink (`bayes_null`) on grey: the null hypothesis wins, so no harm.

In [ ]:
tab = LA.s1_table(nll, post, gd)
print("S1 -> build S2 (drafted rule, n_A = 5):", LA.s1_verdict(tab))
tab[tab.n_b == 80].round(3)

### S1: posterior shape vs $n_A$ (shift stratum, $N_B = 80$)

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 3.3))
p = post[(post.kind == "shift") & (post.n_b == 80) & (post.n_a > 0)]
for d, ls in ((1, "-"), (2, "--")):
    for meth, col in (("pseudo", LA.COLORS["grid_pseudo"]), ("exact", LA.COLORS["grid_exact"])):
        g = p[(p.d == d) & (p.method == meth)].groupby("n_a")
        for ax, y in zip(axs, (g.n_modes.apply(lambda x: (x == 1).mean()), g.err_map.median(), g.entropy.mean(),
                               g.mass_true.mean())):
            ax.plot(y.index, y.values, ls, marker="o", ms=4, color=col, label=f"{meth}, d = {d}")
for ax, t in zip(axs, ("unimodal fraction", "median MAP error |ŝ − s|", "posterior entropy / log G",
                       "posterior mass within 0.05 of s")):
    ax.set_title(t)
    ax.set_xlabel("$n_A$")
axs[1].set_yscale("log")
axs[-1].legend(frameon=False, fontsize=8)
fig.tight_layout()

**How to read.** Each point averages a statistic of the grid posterior over $s$ across the 100 shift pairs at
that $n_A$ (orange = pseudo-likelihood, purple = exact; solid $d = 1$, dashed $d = 2$; orange and purple mostly overlap).

- **Unimodal fraction:** share of pairs whose posterior has exactly one connected region above 5% of its peak (on the
  grid; 8-neighbourhood in 2-D). This is "when is one hypothesis enough".
- **Median MAP error** $\lVert \hat s_{\text{MAP}} - s \rVert$ (log scale). Floor = grid resolution (step 0.002 for
  $d = 1$, 0.01 for $d = 2$); the prior range is 0.4 per axis.
- **Entropy / log G:** 1 = flat over the grid, 0 = all mass on one grid point.
- **Mass within 0.05 of s** ($L_\infty$): the prior alone gives about 0.25 ($d = 1$) and 0.06 ($d = 2$).

What to look for: unimodality jumps between $n_A = 1$ and 5. In $d = 2$ the MAP error stalls near 0.05 and only about
half the mass is near the truth, while prediction is fine (S1 NLL figure). The posterior is concentrated, but along a
direction in which $f$ barely changes, so the error there doesn't matter for prediction (see the 2-D heatmaps).

### S1 examples: posteriors over $s$ ($d = 1$; grey line = true $s$)

In [ ]:
ep = T["ex_post"]
e1 = ep[(ep.d == 1) & (ep.n_b == 80) & (ep.n_a > 0)]
blues = {1: "#9ec5f4", 2: "#5598e7", 5: "#256abf", 10: "#0d366b"}
panels = [("shift", p) for p in sorted(e1.pair.unique())] + [("identity", 0), ("unrelated", 0)]
fig, axs = plt.subplots(2, len(panels), figsize=(3.4 * len(panels), 5.6), sharex=True)
for c, (kind, pr) in enumerate(panels):
    for r, meth in enumerate(("pseudo", "exact")):
        ax = axs[r, c]
        for n, col in blues.items():
            g = e1[(e1.kind == kind) & (e1.pair == pr) & (e1.method == meth) & (e1.n_a == n)].sort_values("s0")
            w = np.exp(g.lw - g.lw.max())
            ax.plot(g.s0, w / w.sum() * len(w) / 0.4, color=col, lw=1.5, label=f"$n_A$={n}")
        st = post[(post.d == 1) & (post.n_b == 80) & (post.kind == kind) & (post.pair == pr)].s_true.iloc[0][0]
        if kind != "unrelated":
            ax.axvline(st, color="#6b6a64", lw=1)
        ax.set_title(f"{kind} pair {pr}, {meth}", fontsize=9)
        ax.set_xlabel("s")
axs[0, 0].set_ylabel("posterior density")
axs[1, 0].set_ylabel("posterior density")
axs[0, -1].legend(frameon=False, fontsize=8)
fig.tight_layout()

**How to read.** One column per example pair: shift pairs, then one identity pair (true $s = 0$) and one
unrelated pair (no true $s$). The curves are the posterior density over $s$ after $n_A = 1, 2, 5, 10$ points (light → dark
blue). Top row = pseudo-likelihood, bottom = exact chain rule. Grey line = true $s$. Heights are not comparable across
panels (a sharper peak is taller).

- **A peak on the grey line** = identified. **Several peaks** = several shifts under which B explains A's few points
  equally well; they disappear as $n_A$ grows. This is the multimodality that one particle cannot represent.
- **Unrelated:** the posterior still concentrates, confidently, on a wrong shift. Posterior *shape* cannot detect
  unrelatedness; the *evidence* relative to the A-only hypothesis can (`bayes_null`).

### S1 examples: posteriors over $s$ ($d = 2$, pseudo; × = true $s$, + = MAP)

In [ ]:
e2 = ep[(ep.d == 2) & (ep.n_b == 80) & (ep.method == "pseudo") & (ep.kind == "shift") & (ep.n_a > 0)]
prs = sorted(e2.pair.unique())
ns = [1, 2, 5, 10]
fig, axs = plt.subplots(max(len(prs), 1), len(ns), figsize=(3.2 * len(ns), 3.0 * max(len(prs), 1)), squeeze=False)
for r, pr in enumerate(prs):
    st = post[(post.d == 2) & (post.n_b == 80) & (post.kind == "shift") & (post.pair == pr)].s_true.iloc[0]
    for c, n in enumerate(ns):
        g = e2[(e2.pair == pr) & (e2.n_a == n)]
        k = int(round(len(g) ** 0.5))
        lw = g.lw.to_numpy().reshape(k, k)
        w = np.exp(lw - lw.max())
        ext = (g.s1.min(), g.s1.max(), g.s0.min(), g.s0.max())
        ax = axs[r, c]
        ax.imshow(w, origin="lower", extent=ext, cmap="Blues", aspect="equal")
        ax.plot(st[1], st[0], "x", color="#e34948", ms=9, mew=2)
        i = int(np.argmax(w))
        ax.plot(g.s1.to_numpy()[i], g.s0.to_numpy()[i], "+", color="#eda100", ms=10, mew=2)
        ax.set_title(f"pair {pr}, $n_A$ = {n}", fontsize=9)
        ax.set_xlabel("$s_2$")
        ax.set_ylabel("$s_1$")
        ax.grid(False)
fig.tight_layout()

**How to read.** The posterior over $(s_1, s_2)$ for three shift pairs (rows) at $n_A = 1, 2, 5, 10$ (columns).
Vertical axis $s_1$, horizontal $s_2$; darker = more mass, normalised per panel to its peak. × = true $s$, + = MAP.

- **$n_A = 1$:** the mass lies on curves. One observation only says "$f_B(x_1 + s) \approx y_1$", which pins $s$ to a
  level set of $f_B$ (shifted by $x_1$): a 1-D curve in the 2-D shift space.
- **More points** give more level sets; their intersection shrinks to a blob near ×. If the MAP lands on another
  branch at small $n_A$ (pair 2), that is a multimodal posterior.
- **An elongated blob** = a direction along which $f$ is flat: $s$ is not identified along it, but predictions don't
  care. This is why S2 should be supervised in function space.

### S1: gradient descent on $s$ through $Q_B$ (from $s = 0$)

Left: share of shift pairs where GD reaches the grid MAP's pseudo log-likelihood (within 0.1 nats), from $s = 0$ (pre-registered) and best of 8 starts (added after the first look). Right ($d = 1$ examples, $n_A = 5$): the pseudo log-likelihood landscape over $s$ and GD's path (dots every 10 steps).

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(17, 3.4))
gs = gd[(gd.n_b == 80) & (gd.kind == "shift")].groupby(["d", "start", "n_a"]).success.mean().reset_index()
for d, ls in ((1, "-"), (2, "--")):
    for start, col in (("zero", LA.COLORS["oracle_q"]), ("best8", LA.COLORS["grid_pseudo"])):
        g = gs[(gs.d == d) & (gs.start == start)]
        axs[0].plot(g.n_a, g.success, ls, marker="o", ms=4, color=col, label=f"{start}, d = {d}")
axs[0].set_title("GD success rate (shift)")
axs[0].set_xlabel("$n_A$")
axs[0].set_ylim(0, 1.05)
axs[0].legend(frameon=False, fontsize=7)
eg = T["ex_gd"]
for ax, pr in zip(axs[1:], sorted(e1.pair.unique())):
    g = e1[(e1.kind == "shift") & (e1.pair == pr) & (e1.method == "pseudo") & (e1.n_a == 5)].sort_values("s0")
    ax.plot(g.s0, g.lw, color=LA.COLORS["grid_pseudo"], lw=1.5)
    tr = eg[(eg.d == 1) & (eg.n_b == 80) & (eg.kind == "shift") & (eg.pair == pr) & (eg.n_a == 5)].sort_values("step")
    sv = np.array([s[0] for s in tr.s])
    yv = np.interp(sv, g.s0, g.lw)
    ax.plot(sv, yv, "o-", color="black", ms=3, lw=0.8)
    st = tr.s_true.iloc[0][0]
    ax.axvline(st, color="#6b6a64", lw=1)
    ax.set_title(f"shift pair {pr}: log L(s), $n_A$ = 5", fontsize=9)
    ax.set_xlabel("s")
fig.tight_layout()

**How to read.**

- **Left:** "success" = GD's final pseudo log-likelihood is within 0.1 nats of the best grid point. It measures
  whether GD found the global optimum, not its distance to the true $s$. Blue = one start at $s = 0$ (the identity
  initialisation an amortised model would use); orange = best of 8 starts, picked by log-likelihood.
- **Right ($d = 1$ examples, $n_A = 5$):** the orange curve is the landscape $\log L(s) = \sum_a \log Q_B(y_a \mid x_a + s)$,
  the log posterior up to a constant. Black dots are GD's iterates every 10 steps from $s = 0$, drawn on the curve at
  their $s$. Grey = true $s$.

What to look for: many local maxima, spaced roughly at the length scale of $f$. GD climbs the nearest hill (pair 2 gets
stuck around $s \approx 0.04$). The global peak is tens of nats above the others, so the *posterior* is essentially a point
mass even though the *landscape* is rugged: inference is a search problem, not an ambiguity problem. That argues for
good proposals or several particles over more local refinement steps.

### Out-of-range probe: $N_B = 200$ (the design note's value; outside `pfn_large`'s $n \le 100$)

Same quantities as above, for $N_B = 80$ vs $200$, shift stratum. Not used for any verdict.

In [ ]:
s = LA.nll_summary(nll)
s = s[(s.kind == "shift")].pivot_table(index=["d", "n_a"], columns=["n_b", "predictor"], values="mean")
cols = [(nb, p) for nb in (80, 200) for p in ("a_only", "oracle_q", "teacher_b", "grid_pseudo", "grid_exact")
        if (nb, p) in s.columns]
s[cols].round(3)

**How to read.** Mean NLL per query (lower is better), shift stratum, $N_B = 80$ vs the out-of-range
$N_B = 200$. Look for whether the larger B context still behaves: it does, and is slightly better. It is not used for any
verdict, because `pfn_large` was trained on contexts with $n \le 100$.

## Verdict

Snapshot `results/ladder/pilot_20261028-143711` (4/4 members DONE, 100 pairs per stratum and member; largest SE of any
mean NLL 0.55 nats, on the unrelated stratum).

- **S0: PASS** (both $d$, $n_A \le 2$). The oracle particle beats A-only by 3.3–5.4 nats/query ($d = 1$) and 2.8–4.1
  ($d = 2$), CI lower bounds ≥ 2.6; it matches the B-space teacher within 0.01 and the A-space teacher within 0.07. The query
  path, caching and scale handling work. Naive pooling is worse than A-only (registration is necessary).
- **S1 → build S2: GO** ($n_A = 5$, $N_B = 80$). `grid_pseudo` closes 95% ($d = 1$) and 88% ($d = 2$) of the A-only → oracle
  gap; unimodal posteriors 98% / 97%. The pseudo-likelihood costs almost nothing against the exact chain rule
  (≤ 0.04 nats at $n_A \le 5$).
- For S2's design: (i) one hypothesis is not enough at $n_A \le 2$ (unimodal 56–84%); (ii) the landscape is rugged: GD from
  $s = 0$ reaches the MAP in 43–61% of pairs, best of 8 starts in 89–99%, so proposals / several particles matter more than
  local refinement; (iii) in $d = 2$, $s$ is only partly identifiable (median MAP error 0.05 at $n_A = 5$, ridges along
  flat directions of $f$) while predictions are fine: supervise in function space, not $\lVert s - s_{\text{true}} \rVert^2$
  alone; (iv) without a null hypothesis the unrelated stratum costs +5 to +13 nats/query vs A-only; `bayes_null` is within 0.4 at $n_A = 1$ and 0.03 from $n_A = 5$.
- Probe $N_B = 200$ (outside the PFN's range): the oracle is slightly better (−4.35 vs −4.24, $d = 1$), no breakdown.
